In [ ]:
# FONT_DIR = r"C:\Windows\Fonts" ## OR BELOW
FONT_DIR = r"fonts" ## OR ABOVE

from fontTools.ttLib import TTFont
from fontTools.pens.svgPathPen import SVGPathPen
from fontTools.pens.boundsPen import BoundsPen
import svgpathtools as spt
import os
import re

# ---- CONFIG ----

CHARS = {
    "Arabic":  ["٠","١","٢","٣","٤","٥","٦","٧","٨","٩"],
    "Hindi":   ["०","१","२","३","४","५","६","७","८","९"],
    "Mandarin":["零","一","二","三","四","五","六","七","八","九"],
}


OUTPUT_DIR = "digits_output_svg_normalized-final"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CANVAS_W = 160
CANVAS_H = 200

PATH_RE = re.compile(r'd="([^"]+)"')

# ---- HELPERS ----

def font_supports_all(font_path, chars):
    try:
        tt = TTFont(font_path)
    except:
        return False

    cmap = tt.getBestCmap()
    if not cmap:
        return False

    return all(ord(ch) in cmap for ch in chars)


def extract_glyph_path(font, char):
    cmap = font.getBestCmap()
    if not cmap:
        return None

    cp = ord(char)
    if cp not in cmap:
        return None

    glyph_name = cmap[cp]
    glyph_set = font.getGlyphSet()

    if glyph_name not in glyph_set:
        return None

    glyph = glyph_set[glyph_name]

    # Compute bounding box
    bp = BoundsPen(glyph_set)
    glyph.draw(bp)

    if bp.bounds is None:
        bp = BoundsPen(glyph_set)
        glyph.draw(bp)

    if bp.bounds is None:
        return None

    xmin, ymin, xmax, ymax = bp.bounds

    # Convert glyph to SVG path
    pen = SVGPathPen(glyph_set)
    glyph.draw(pen)
    path_data = pen.getCommands()

    if not path_data.strip():
        return None

    return path_data, (xmin, ymin, xmax, ymax)


def extract_all_glyphs(font_path, chars):
    """Return dict: char → (path_data, bbox) or None if unsupported."""
    try:
        temp = TTFont(font_path)
        num_fonts = temp.reader.numFonts if hasattr(temp.reader, "numFonts") else 1
    except:
        return None

    for idx in range(num_fonts):
        try:
            font = TTFont(font_path, fontNumber=idx)
        except:
            continue

        glyphs = {}
        for ch in chars:
            result = extract_glyph_path(font, ch)
            if result is None:
                glyphs = None
                break
            glyphs[ch] = result

        if glyphs:
            return glyphs

    return None


# ---- MAIN LOOP ----

for font_file in os.listdir(FONT_DIR):
    if not font_file.lower().endswith((".ttf", ".otf", ".ttc")):
        continue

    font_path = os.path.join(FONT_DIR, font_file)

    for lang, chars in CHARS.items():

        # Skip fonts that don't support this language
        if not font_supports_all(font_path, chars):
            continue

        print(f"Extracting + Normalizing {lang} from {font_file}")

        glyphs = extract_all_glyphs(font_path, chars)
        if glyphs is None:
            continue

        # ---- Compute union bounding box ----
        bboxes = [bbox for (_, bbox) in glyphs.values()]
        union_xmin = min(b[0] for b in bboxes)
        union_ymin = min(b[1] for b in bboxes)
        union_xmax = max(b[2] for b in bboxes)
        union_ymax = max(b[3] for b in bboxes)

        union_w = union_xmax - union_xmin
        union_h = union_ymax - union_ymin

        # ---- Compute normalization transform ----
        cx = (union_xmin + union_xmax) / 2
        cy = (union_ymin + union_ymax) / 2

        scale_x = CANVAS_W / union_w
        scale_y = CANVAS_H / union_h
        s = min(scale_x, scale_y)

        transform = (
            f"translate({CANVAS_W/2},{CANVAS_H/2}) "
            f"scale({s}, {-s}) "
            f"translate({-cx}, {-cy})"
        )

        # ---- Output folder ----
        out_dir = os.path.join(OUTPUT_DIR, lang, font_file)
        os.makedirs(out_dir, exist_ok=True)

        # ---- Write normalized SVGs ----
        for idx, ch in enumerate(chars):
            path_data, _ = glyphs[ch]

            svg = f"""<svg xmlns="http://www.w3.org/2000/svg"
     width="{CANVAS_W}" height="{CANVAS_H}"
     viewBox="0 0 {CANVAS_W} {CANVAS_H}">
  <g transform="{transform}">
    <path d="{path_data}" fill="black"/>
  </g>
</svg>
"""

            out_path = os.path.join(out_dir, f"{idx}_{lang}.svg")
            with open(out_path, "w", encoding="utf-8") as f:
                f.write(svg)

print("Done — ALL languages extracted + normalized.")
